In [8]:
import pandas as pd
df = pd.read_csv("train_original.csv")
df["label"].unique()

array(['sadness', 'neutral', 'fear', 'happiness', 'anger', 'disgust',
       'surprize', nan], dtype=object)

In [9]:
df["label"] = df["label"].str.lower()

df["label"] = df["label"].replace({"surprize": "surprise"})

df = df.dropna(subset=["label"])

print(df["label"].unique())

['sadness' 'neutral' 'fear' 'happiness' 'anger' 'disgust' 'surprise']


In [3]:
df.head()

,text,label
0,خیلی ناراحت شدم وقتی خبر بدی شنیدم,sadness
1,یک فرشته به خواب یکنفر میاد و بهش میگه خدا گفت...,neutral
2,ترسیدم چون صدای عجیبی شنیدم,fear
3,اونقدر حواسمون بود که بقیه ناراحت نشن که الان ...,sadness
4,خیلی راحته که کدوم جالش,neutral


In [10]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

label_mapping = {
    "happiness": 0,
    "sadness": 1,
    "surprise": 2,
    "anger": 3,
    "disgust": 4,
    "fear": 5,
    "neutral": 6
}

df["label_id"] = df["label"].map(label_mapping)

df = df.dropna(subset=["label_id"])

df["label_id"] = df["label_id"].astype(int)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["label_id"]),
    y=df["label_id"]
)

# Convert to dictionary for Keras
class_weights = {i: weight for i, weight in enumerate(class_weights)}

print("Class weights:", class_weights)


Class weights: {0: 1.2886753246753246, 1: 1.023644466452092, 2: 1.3611896073966363, 3: 0.8038691488844603, 4: 1.2634071810542398, 5: 1.1824681824681824, 6: 0.613018014678627}


In [11]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# Train-test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label_id"],
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

# Convert to string and to list, handle NaN if any
train_texts = train_texts.fillna("").astype(str).tolist()
test_texts = test_texts.fillna("").astype(str).tolist()

# Tokenization
model_name = "HooshvareLab/bert-base-parsbert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


In [6]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 51.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 37.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 61.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 43.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [transformers] [transformers]ub]


In [12]:
import tensorflow as tf

# Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).shuffle(1000).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_labels
)).batch(16)


2025-10-25 15:25:46.845546: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-25 15:25:46.878941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-25 15:25:46.901898: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-25 15:25:46.909227: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-25 15:25:46.941912: I tensorflow/core/platform/cpu_feature_guar

In [13]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=7
)

tf_model.h5:   0%|          | 0.00/963M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at HooshvareLab/bert-base-parsbert-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
from transformers import create_optimizer

batch_size = 16
num_epochs = 5
batches_per_epoch = len(train_texts) // batch_size
total_train_steps = batches_per_epoch * num_epochs

optimizer, lr_schedule = create_optimizer(
    init_lr=2e-5,
    num_train_steps=total_train_steps,
    num_warmup_steps=int(0.1 * total_train_steps),
)


'+ptx85' is not a recognized feature for this target (ignoring feature)
''+ptx85' is not a recognized feature for this target (ignoring feature)
+ptx85' is not a recognized feature for this target'+ptx85 (ignoring feature)
' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

In [15]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy

model.compile(
    optimizer=optimizer,
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=[SparseCategoricalAccuracy("accuracy")]
)


In [16]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=5,
    class_weight=class_weights,
    callbacks=[early_stopping]
)


Epoch 1/5


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

   1/2481 [..............................] - ETA: 48:14:57 - loss: 1.9832 - accuracy: 0.1875

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


   2/2481 [..............................] - ETA: 15:18 - loss: 2.1092 - accuracy: 0.1875   

...

1275/2481 [==============>...............] - ETA: 1:39 - loss: 0.6379 - accuracy: 0.7531

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


2272/2481 [==========================>...] - ETA: 17s - loss: 0.5686 - accuracy: 0.7786 

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


2481/2481 [==============================] - 232s 93ms/step - loss: 0.5663 - accuracy: 0.7792 - val_loss: 1.7378 - val_accuracy: 0.4827


In [17]:
import numpy as np
from sklearn.metrics import classification_report, f1_score, accuracy_score

preds = model.predict(test_dataset).logits
y_pred = np.argmax(preds, axis=1)

accuracy = accuracy_score(test_labels, y_pred)
f1 = f1_score(test_labels, y_pred, average='weighted')

print("Accuracy:", round(accuracy, 3))
print("Weighted F1-score:", round(f1, 3))
print("\nClassification Report:\n")
print(classification_report(test_labels, y_pred, digits=3))


621/621 [==============================] - 29s 40ms/step
Accuracy: 0.513
Weighted F1-score: 0.515

Classification Report:

              precision    recall  f1-score   support

           0      0.608     0.552     0.579      1100
           1      0.478     0.514     0.495      1385
           2      0.684     0.554     0.613      1041
           3      0.495     0.524     0.509      1763
           4      0.591     0.479     0.529      1122
           5      0.524     0.602     0.560      1199
           6      0.415     0.437     0.426      2313

    accuracy                          0.513      9923
   macro avg      0.542     0.523     0.530      9923
weighted avg      0.521     0.513     0.515      9923



In [19]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load test data
df_test = pd.read_csv("test_file.csv")

# map raw dataset labels to canonical labels
raw_to_canonical = {
    'HAPPY': 'happiness',
    'SAD': 'sadness',
    'OTHER': 'neutral',
    'ANGRY': 'anger',
    'SURPRISE': 'surprise',   # fixed typo: surprize → surprise
    'HATE': 'disgust',
    'FEAR': 'fear',
}
df_test["label"] = df_test["label"].map(raw_to_canonical)

# map canonical labels to integer IDs
label_to_id = {
    "happiness": 0,
    "sadness": 1,
    "surprise": 2,
    "anger": 3,
    "disgust": 4,
    "fear": 5,
    "neutral": 6,
}
df_test["label_id"] = df_test["label"].map(label_to_id)

# Drop rows where label wasn't mapped
df_test = df_test.dropna(subset=["label_id"])
df_test["label_id"] = df_test["label_id"].astype(int)

# Prepare texts & labels
external_test_texts = df_test["text"].astype(str).tolist()
external_test_labels = df_test["label_id"].tolist()

# Tokenization
external_encodings = tokenizer(
    external_test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

external_dataset = tf.data.Dataset.from_tensor_slices((
    dict(external_encodings),
    external_test_labels
)).batch(16)

# Predict
external_preds = model.predict(external_dataset).logits
external_y_pred = np.argmax(external_preds, axis=1)

# Evaluate
external_accuracy = accuracy_score(external_test_labels, external_y_pred)
external_f1 = f1_score(external_test_labels, external_y_pred, average='weighted')

print(" External Test Evaluation:")
print("Accuracy:", round(external_accuracy, 3))
print("Weighted F1-score:", round(external_f1, 3))
print("\nClassification Report:")
print(classification_report(external_test_labels, external_y_pred, digits=3))


72/72 [==============================] - 9s 38ms/step
 External Test Evaluation:
Accuracy: 0.56
Weighted F1-score: 0.552

Classification Report:
              precision    recall  f1-score   support

           0      0.822     0.571     0.674       275
           1      0.522     0.756     0.618       262
           2      0.676     0.331     0.444       145
           3      0.497     0.552     0.523       154
           4      0.360     0.138     0.200        65
           5      0.500     0.754     0.601        57
           6      0.461     0.544     0.499       193

    accuracy                          0.560      1151
   macro avg      0.548     0.521     0.508      1151
weighted avg      0.589     0.560     0.552      1151

